In [ ]:
# Check the cluster
# v2 
attached_cluster_name = spark.conf.get(
    "spark.databricks.clusterUsageTags.clusterName", ""
)
if not attached_cluster_name.endswith("uc_support") and not (
    attached_cluster_name.startswith("bfdw_")
    and "compute_uc_jobs" in attached_cluster_name
):
    raise Exception(
        "This notebook is being executed in an incorrect cluster. Please attach it to the *uc_support cluster or one of the bfdw_*compute_uc_jobs* clusters"
    )
else:
    print(f"Cluster is: {attached_cluster_name}")

2.Code to select catolog based on the workspace

In [ ]:
workspace_catalogs = [
    e.catalog.lower()
    for e in spark.sql(f"SHOW CATALOGS").collect()
    if e.catalog not in ["main", "samples", "system", "__databricks_internal"]
]
print(f"Catalogs in the workspace: {workspace_catalogs}")
banfield_catalogs = ["banfield_catalogdev", "banfield_catalogtst", "banfield_catalog"]
target_catalog0 = [e for e in banfield_catalogs if e in workspace_catalogs]
if (not target_catalog0) or len(target_catalog0) != 1:
    raise Exception(
        f"Expecting any one of the active banfield catalog but received {len(target_catalog0)}; Banfield catalog names: {banfield_catalogs}"
    )
spark.conf.set("catlg.banfield_catalog", target_catalog0[0])
print(f"catlg.banfield_catalog: {target_catalog0[0]}")
 
bf_vwmvhcores = ["bf_vwmvhcoredev", "bf_vwmvhcoretst", "bf_vwmvhcore"]
target_catalog1 = [e for e in bf_vwmvhcores if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog1) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog1)}; bf_vwmvhcore names: {bf_vwmvhcores}"
    )
spark.conf.set("catlg.bf_vwmvhcore", target_catalog1[0])
print(f"catlg.bf_vwmvhcore: {target_catalog1[0]}")


bf_vwedhs = ["bf_vwedhdev", "bf_vwedhtst", "bf_vwedh"]
target_catalog2 = [e for e in bf_vwedhs if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog2) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog2)}; bf_vwmvhcore names: {bf_vwedhs}"
    )
spark.conf.set("catlg.bf_vwedh", target_catalog2[0])
print(f"catlg.bf_vwedh: {target_catalog2[0]}")

bf_vwvoyagers = ["bf_vwvoyagerdev", "bf_vwvoyagertst", "bf_vwvoyager"]
target_catalog3 = [e for e in bf_vwvoyagers if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog3) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog3)}; bf_vwvoyager names: {bf_vwvoyagers}"
    )
spark.conf.set("catlg.bf_vwvoyager", target_catalog3[0])
print(f"catlg.bf_vwvoyager: {target_catalog3[0]}")

3. Create table Bronze Layer

In [ ]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdcurrenthosp (
hosp_num STRING,
    frmr_hosp_num STRING,
    hosp_typ STRING,
    petsmart_vet_srvcs_flg STRING NOT NULL,
    hosp_abbrv STRING,
    store_num STRING,
    hosp_long_nam STRING,
    hosp_short_nam STRING,
    hosp_region STRING,

    regional_vp_full_nam STRING,
    regional_vp_first_nam STRING,
    regional_vp_last_nam STRING,

    field_drctr_area STRING,
    field_drctr_full_nam STRING,
    field_drctr_first_nam STRING,
    field_drctr_last_nam STRING,

    field_trainer_full_nam STRING,
    field_trainer_first_nam STRING,
    field_trainer_last_nam STRING,

    medical_drctr_full_nam STRING,
    medical_drctr_first_nam STRING,
    medical_drctr_last_nam STRING,

    prctc_cnsltnt_full_nam STRING,
    prctc_cnsltnt_first_nam STRING,
    prctc_cnsltnt_last_nam STRING,

    cos_full_nam STRING,
    cos_first_nam STRING,
    cos_last_nam STRING,

    shopping_center_nam STRING,
    hosp_cross_streets STRING,
    hosp_street_addr1 STRING,
    hosp_street_addr2 STRING,
    hosp_city STRING,
    hosp_cnty STRING,
    hosp_state STRING,
    hosp_cntry STRING,
    hosp_postal_cd STRING,

    hosp_phone_num1 STRING,
    hosp_phone_num2 STRING,
    hosp_phone_num3 STRING,
    hosp_phone_num4 STRING,
    hosp_phone_num5 STRING,
    hosp_phone_num6 STRING,
    hosp_phone_num7 STRING,
    hosp_phone_num8 STRING,

    hosp_fax_num STRING,

    hosp_open_dt DATE,
    hosp_open_flg STRING NOT NULL,
    store_open_dt DATE,
    hosp_close_dt DATE,
    hosp_cnvrtd_dt DATE,
    hosp_relo_dt DATE,

    hosp_time_zone_descr STRING,
    hosp_updtd_hrs_dt DATE,

    mon_open_hr TIMESTAMP,
    mon_close_hr TIMESTAMP,
    tue_open_hr TIMESTAMP,
    tue_close_hr TIMESTAMP,
    wed_open_hr TIMESTAMP,
    wed_close_hr TIMESTAMP,
    thu_open_hr TIMESTAMP,
    thu_close_hr TIMESTAMP,
    fri_open_hr TIMESTAMP,
    fri_close_hr TIMESTAMP,
    sat_open_hr TIMESTAMP,
    sat_close_hr TIMESTAMP,
    sun_open_hr TIMESTAMP,
    sun_close_hr TIMESTAMP,

    tot_wk_hrs_open INT,
    hosp_days_not_open STRING,
    kennel_cnt BIGINT,

    dw_create_dt TIMESTAMP NOT NULL,
    dw_begin_eff_dt DATE NOT NULL,
    dw_end_eff_dt DATE NOT NULL,
    dw_curr_row_ind INT NOT NULL,
    dw_deleted_ind INT NOT NULL,
    dw_job_id BIGINT NOT NULL,
    dw_load_dt TIMESTAMP NOT NULL,
    record_status string )
  
using delta

tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')


4. Create table for Silver Layer

In [ ]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrenthosp (
  hosp_num STRING,
    frmr_hosp_num STRING,
    hosp_typ STRING,
    petsmart_vet_srvcs_flg STRING NOT NULL,
    hosp_abbrv STRING,
    store_num STRING,
    hosp_long_nam STRING,
    hosp_short_nam STRING,
    hosp_region STRING,

    regional_vp_full_nam STRING,
    regional_vp_first_nam STRING,
    regional_vp_last_nam STRING,

    field_drctr_area STRING,
    field_drctr_full_nam STRING,
    field_drctr_first_nam STRING,
    field_drctr_last_nam STRING,

    field_trainer_full_nam STRING,
    field_trainer_first_nam STRING,
    field_trainer_last_nam STRING,

    medical_drctr_full_nam STRING,
    medical_drctr_first_nam STRING,
    medical_drctr_last_nam STRING,

    prctc_cnsltnt_full_nam STRING,
    prctc_cnsltnt_first_nam STRING,
    prctc_cnsltnt_last_nam STRING,

    cos_full_nam STRING,
    cos_first_nam STRING,
    cos_last_nam STRING,

    shopping_center_nam STRING,
    hosp_cross_streets STRING,
    hosp_street_addr1 STRING,
    hosp_street_addr2 STRING,
    hosp_city STRING,
    hosp_cnty STRING,
    hosp_state STRING,
    hosp_cntry STRING,
    hosp_postal_cd STRING,

    hosp_phone_num1 STRING,
    hosp_phone_num2 STRING,
    hosp_phone_num3 STRING,
    hosp_phone_num4 STRING,
    hosp_phone_num5 STRING,
    hosp_phone_num6 STRING,
    hosp_phone_num7 STRING,
    hosp_phone_num8 STRING,

    hosp_fax_num STRING,

    hosp_open_dt DATE,
    hosp_open_flg STRING NOT NULL,
    store_open_dt DATE,
    hosp_close_dt DATE,
    hosp_cnvrtd_dt DATE,
    hosp_relo_dt DATE,

    hosp_time_zone_descr STRING,
    hosp_updtd_hrs_dt DATE,

    mon_open_hr TIMESTAMP,
    mon_close_hr TIMESTAMP,
    tue_open_hr TIMESTAMP,
    tue_close_hr TIMESTAMP,
    wed_open_hr TIMESTAMP,
    wed_close_hr TIMESTAMP,
    thu_open_hr TIMESTAMP,
    thu_close_hr TIMESTAMP,
    fri_open_hr TIMESTAMP,
    fri_close_hr TIMESTAMP,
    sat_open_hr TIMESTAMP,
    sat_close_hr TIMESTAMP,
    sun_open_hr TIMESTAMP,
    sun_close_hr TIMESTAMP,

    tot_wk_hrs_open INT,
    hosp_days_not_open STRING,
    kennel_cnt BIGINT,

    dw_create_dt TIMESTAMP NOT NULL,
    dw_begin_eff_dt DATE NOT NULL,
    dw_end_eff_dt DATE NOT NULL,
    dw_curr_row_ind INT NOT NULL,
    dw_deleted_ind INT NOT NULL,
    dw_job_id BIGINT NOT NULL,
    dw_load_dt TIMESTAMP NOT NULL,
    record_status string 
  
using delta

tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')


5. Create View for Gold Layer

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_gold.cmn_tbcmdcurrenthosp

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrenthosp a where 1 =1 

6. Create View for bfdw_date_quality for silver layer

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdcurrenthosp_silver_primary_key_exceptions

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cmn_tbcmdcurrenthosp a where 1 !=1 


7. Create View for bfdw_data_quality for Bronze layers

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.cmn_tbcmdcurrenthosp_bronze_record_status_exceptions

as
Select *  
from  ${catlg.banfield_catalog}.bfdw_bronze.cmn_tbcmdcurrenthosp
where record_status != 'valid';